# Titanic
## Score: .78947


In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}

In [9]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    fare_edges = np.unique(np.quantile(tr["Fare"], q=np.linspace(0.0, 1.0, 6)))
    if len(fare_edges) < 3:
        fare_edges = np.array([tr["Fare"].min(), tr["Fare"].median(), tr["Fare"].max()])

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["IsChild"] = (out["Age"] < 16).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["CabinCount"] = out["Cabin"].fillna("").astype(str).str.split().str.len().clip(lower=0, upper=3)
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeClass"] = out["Age"] * out["Pclass"]
        out["FareClass"] = out["Fare"] / out["Pclass"].clip(lower=1)
        out["NameLength"] = out["Name"].str.len().fillna(0).astype(int)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["FareBin"] = pd.cut(out["Fare"], bins=fare_edges, include_lowest=True, duplicates="drop").astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["FamilyId"] = (out["Surname"].fillna("UNK") + "_" + out["FamilySize"].astype(str)).astype(str)
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "FareBin", "Surname", "FamilyId", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=20.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "FareBin", "Surname", "FamilyId", "TicketPrefix"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "FareBin", "Surname", "FamilyId", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=18.0)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "IsChild", "TicketGroupSize", "FarePerPerson", "HasCabin",
    "AgeClass", "FareClass", "CabinCount", "NameLength"
] + [c + "_te" for c in te_cols]

imp = SimpleImputer(strategy="median")
X_hgb = imp.fit_transform(X_te[hgb_features])
X_test_hgb = imp.transform(X_test_te[hgb_features])

X_lgb = pd.get_dummies(X_te, columns=cat_cols, drop_first=False)
X_test_lgb = pd.get_dummies(X_test_te, columns=cat_cols, drop_first=False)
X_lgb, X_test_lgb = X_lgb.align(X_test_lgb, join="left", axis=1, fill_value=0.0)

# LightGBM rejects some special characters in feature names.
_safe_cols = (
    pd.Index(X_lgb.columns)
    .astype(str)
    .str.replace(r"[^0-9A-Za-z_]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)
X_lgb.columns = _safe_cols
X_test_lgb.columns = _safe_cols

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_cb1 = np.zeros(len(y), dtype=float)
oof_cb2 = np.zeros(len(y), dtype=float)
oof_hgb = np.zeros(len(y), dtype=float)
oof_lgb = np.zeros(len(y), dtype=float)

p_cb1_t = np.zeros(len(X_test), dtype=float)
p_cb2_t = np.zeros(len(X_test), dtype=float)
p_hgb_t = np.zeros(len(X_test), dtype=float)
p_lgb_t = np.zeros(len(X_test), dtype=float)

fold_acc_cb1 = []
fold_acc_cb2 = []
fold_acc_hgb = []
fold_acc_lgb = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    Xc_tr, Xc_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    Xh_tr, Xh_va = X_hgb[tr_idx], X_hgb[va_idx]
    Xl_tr, Xl_va = X_lgb.iloc[tr_idx], X_lgb.iloc[va_idx]

    cb1 = CatBoostClassifier(
        iterations=24000,
        learning_rate=0.014,
        depth=7,
        l2_leaf_reg=3.0,
        random_seed=SEED + fold,
        thread_count=1,
        verbose=False,
    )
    cb1.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=400,
        verbose=False,
    )

    cb2 = CatBoostClassifier(
        iterations=30000,
        learning_rate=0.010,
        depth=9,
        l2_leaf_reg=4.5,
        random_seed=SEED + 100 + fold,
        thread_count=1,
        verbose=False,
    )
    cb2.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=500,
        verbose=False,
    )

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.018,
        max_iter=1800,
        max_depth=8,
        min_samples_leaf=5,
        l2_regularization=0.30,
        random_state=SEED + fold,
    )
    hgb.fit(Xh_tr, y_tr)

    lgbm = lgb.LGBMClassifier(
        n_estimators=2500,
        learning_rate=0.015,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=12,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=0.2,
        random_state=SEED + fold,
        objective="binary",
    )
    lgbm.fit(
        Xl_tr,
        y_tr,
        eval_set=[(Xl_va, y_va)],
        eval_metric="binary_logloss",
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )

    p1 = cb1.predict_proba(Xc_va)[:, 1]
    p2 = cb2.predict_proba(Xc_va)[:, 1]
    p3 = hgb.predict_proba(Xh_va)[:, 1]
    p4 = lgbm.predict_proba(Xl_va)[:, 1]

    oof_cb1[va_idx] = p1
    oof_cb2[va_idx] = p2
    oof_hgb[va_idx] = p3
    oof_lgb[va_idx] = p4

    p_cb1_t += cb1.predict_proba(X_test)[:, 1] / cv.n_splits
    p_cb2_t += cb2.predict_proba(X_test)[:, 1] / cv.n_splits
    p_hgb_t += hgb.predict_proba(X_test_hgb)[:, 1] / cv.n_splits
    p_lgb_t += lgbm.predict_proba(X_test_lgb)[:, 1] / cv.n_splits

    fold_acc_cb1.append(accuracy_score(y_va, (p1 >= 0.5).astype(int)))
    fold_acc_cb2.append(accuracy_score(y_va, (p2 >= 0.5).astype(int)))
    fold_acc_hgb.append(accuracy_score(y_va, (p3 >= 0.5).astype(int)))
    fold_acc_lgb.append(accuracy_score(y_va, (p4 >= 0.5).astype(int)))

print("cv_base_acc", {
    "cb1": round(float(np.mean(fold_acc_cb1)), 5),
    "cb2": round(float(np.mean(fold_acc_cb2)), 5),
    "hgb": round(float(np.mean(fold_acc_hgb)), 5),
    "lgb": round(float(np.mean(fold_acc_lgb)), 5),
})

stack_oof = np.column_stack([oof_cb1, oof_cb2, oof_hgb, oof_lgb])
rng = np.random.default_rng(SEED)
weights = rng.dirichlet(np.array([1.3, 1.1, 0.9, 1.0]), size=7000)

best_acc = -1.0
best_w = weights[0]
best_t = 0.5

for w in weights:
    p = stack_oof @ w
    for t in np.linspace(0.40, 0.66, 131):
        a = accuracy_score(y, (p >= t).astype(int))
        if a > best_acc:
            best_acc = a
            best_w = w.copy()
            best_t = float(t)

best_w1, best_w2, best_w3, best_w4 = [float(v) for v in best_w]
oof_blend = stack_oof @ best_w
print(
    "cv_blend_acc",
    round(best_acc, 5),
    "w",
    [round(best_w1, 3), round(best_w2, 3), round(best_w3, 3), round(best_w4, 3)],
    "t",
    round(best_t, 3),
)
print(confusion_matrix(y, (oof_blend >= best_t).astype(int)))

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000335 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 914
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [11]:
blend_t = best_w1 * p_cb1_t + best_w2 * p_cb2_t + best_w3 * p_hgb_t + best_w4 * p_lgb_t
submission = pd.DataFrame({"PassengerId": pid, "Survived": (blend_t >= best_t).astype(int)})
submission.to_csv(ROOT / "submission.csv", index=False)
print("cv_ensemble", "w", [round(best_w1, 3), round(best_w2, 3), round(best_w3, 3), round(best_w4, 3)], "t", round(best_t, 3), ROOT / "submission.csv")
submission.head(10)

cv_ensemble w [0.068, 0.21, 0.009, 0.713] t 0.44 c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
